# Chapter 12 &mdash; The PDA Edge Label: `input , pop ; push`

**Concept 2 of the Chapter 12 decomposition:** *The PDA Edge Label: `input , pop ; push`*

Read one symbol or $\varepsilon$; pop one stack symbol or nothing; push a string or nothing.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12/Concept-PDA-Edge-Label/Concept-PDA-Edge-Label.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.AnimatePDA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Jove's PDA edge label has three fields:

```
State : input , pop ; push -> NextState
```

* **input** &mdash; one symbol, or `''` for an $\varepsilon$ move (read nothing);
* **pop** &mdash; the symbol that must be on top, which is removed;
* **push** &mdash; a *string* pushed in its place, **leftmost symbol ends up on top**;
  `''` pushes nothing.

Two idioms cover most designs:

* `a , X ; YX` &mdash; **push** $Y$ (pop $X$ and put it straight back, with $Y$ above);
* `a , X ; ''` &mdash; **pop** $X$.

`a , X ; X` is a **peek**: it checks the top without changing the stack.

## 2. Definitions

### The three idioms, in one machine

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

In [ ]:
Idioms = md2mc('''PDA
I : a , # ; A#   -> I    !! PUSH: pop #, push back A then #  (A ends up on top)
I : b , A ; ''   -> I    !! POP : remove the A
I : c , # ; #    -> I    !! PEEK: require # on top, leave it there
I : '' , # ; #   -> F
''')

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

### Inspecting Delta

In [ ]:
def show_delta(P):
    for (q, inp, pop), outs in sorted(P["Delta"].items()):
        for (q2, push) in outs:
            print("  %-3s : %-3r , %-3s ; %-4s -> %s"
                  % (q, inp, pop, push if push else "''", q2))

## 3. Tests

The three idioms, as Jove stores them.

In [ ]:
show_delta(Idioms)

**Push:** the leftmost pushed symbol ends up on top.

In [ ]:
surv, paths, visited = run_pda('a', Idioms, STKMAX=6)
stacks = sorted({st for (_, _, st) in visited})
print("stacks seen after reading 'a' :", stacks)
assert 'A#' in stacks
print("\n'A#' -- A is on top, # underneath.  Leftmost pushed = topmost.")

**Pop:** `a` then `b` returns the stack to just `#`.

In [ ]:
print("accepts 'ab' ?", pda_accepts(Idioms, 'ab', STKMAX=6))
print("accepts 'a'  ?", pda_accepts(Idioms, 'a', STKMAX=6))
assert pda_accepts(Idioms, 'ab', STKMAX=6)
assert not pda_accepts(Idioms, 'a', STKMAX=6)
print("\n'a' alone leaves A on the stack, so the epsilon move to F cannot fire.")

**Peek:** `c` requires `#` on top and leaves it there.

In [ ]:
print("accepts 'c'   ?", pda_accepts(Idioms, 'c', STKMAX=6))
print("accepts 'ac'  ?", pda_accepts(Idioms, 'ac', STKMAX=6))
assert pda_accepts(Idioms, 'c', STKMAX=6)
assert not pda_accepts(Idioms, 'ac', STKMAX=6)
print("\nafter 'a' the top is A, not #, so the peek fails.")

An $\varepsilon$ input move reads nothing &mdash; useful for finishing up.

In [ ]:
eps_edges = [k for k in Idioms["Delta"] if k[1] == '']
print("epsilon-input edges :", eps_edges)
assert eps_edges
print("\nthe accept move fires without consuming input, once the stack is clean.")

## 4. Animation

The three idioms drawn: push, pop and peek.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimatePDA import *
AnimatePDA(Idioms, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Write an edge that pushes **two** symbols at once. Which ends up on top?
2. What does `a , X ; YZ` do, exactly?
3. Can a PDA pop two symbols in one move? How would you simulate it?

In [ ]:
# Your work for the exercises above.